# 08 — Shuffled-Amplitude Null and Recovery Under Correlated Noise (SPARC only)

**Project:** *Is the RAR Universal? Correlated-error-aware constraints on the scatter and acceleration scale of the radial acceleration relation in SPARC and independent HI samples*

This notebook completes **proposal Section 4.6** (mocks "with realistic radial sampling and correlated noise"; recovery and false-positive rates calibrated on zero-injection mocks). It answers two referee points on Notebook 06:

* **C1 — shuffled-amplitude null.** Notebook 06's correlated-noise null (A1) took each galaxy's noise amplitude A_i from the Model D fit. A_i correlates with baryonic mass (r ≈ 0.62), so A1 may itself carry population information, which makes the calibration partly circular. Here each mock **randomly permutes A_i across galaxies** (each galaxy keeps its own length scale ℓ_i, which is tied to its radial sampling). The noise level is realistic, but it has no link to mass, surface brightness or gas fraction. The universal RAR truth is the same as A1's (Model D medians).
* **C2 — recovery under correlated noise.** Notebook 06 injected slopes on a white-noise base only. Here the measured Notebook 05 slopes are injected at 1× and 2× **on top of A1-type correlated noise** (unshuffled A_i). Power is reported two ways: with the naive Bonferroni rule, and after calibration against the A1 null of Notebook 06 using a Student-t cutoff.

**Unchanged:** the model, priors, likelihood, covariates, decision rule (Notebook 05), generator and mock-fit sampler (Notebook 06). Mock fits use 2 chains × (500 + 750). A fit counts only if R-hat < 1.05, bulk ESS ≥ 100 and there are 0 divergences.

**Runtime.** About 1.5–2 min per fit on Colab. The default plan has 20 C1 mocks × 3 fits and 12 C2 fits, i.e. 72 fits, roughly 2–2.5 h. Results are written after every fit, and an interrupted run resumes with `RAR_RESUME_FROM`.

## 2. Environment and Configuration

In [ ]:
import os, sys, importlib.util, subprocess
N_CPU = os.cpu_count() or 1
os.environ["XLA_FLAGS"] = f"--xla_force_host_platform_device_count={max(1, min(N_CPU, 8))}"
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.isdir("/kaggle/working")
REQUIRED = {"jax": "jax", "numpyro": "numpyro", "xarray": "xarray", "h5netcdf": "h5netcdf", "h5py": "h5py",
            "scipy": "scipy", "matplotlib": "matplotlib", "pandas": "pandas", "mpmath": "mpmath"}
missing = [p for m, p in REQUIRED.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import re, json, time, glob, hashlib, logging, platform, warnings, resource, datetime as dt
from pathlib import Path
import numpy as np, pandas as pd, scipy, scipy.stats as sst, xarray as xr
import matplotlib, matplotlib.pyplot as plt
import jax, jax.numpy as jnp
from jax.scipy.linalg import cholesky as jcholesky, solve_triangular
import numpyro, numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from numpyro.infer.util import log_density, unconstrain_fn
from jax.scipy.special import ndtr, ndtri
from numpyro.diagnostics import split_gelman_rubin, effective_sample_size
numpyro.enable_x64()
numpyro.set_host_device_count(max(1, min(N_CPU, 8)))

RUN_TIMESTAMP = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
            "jax": jax.__version__, "numpyro": numpyro.__version__, "xarray": xr.__version__, "matplotlib": matplotlib.__version__,
            "cpu_count": N_CPU, "jax_devices": len(jax.devices()), "jax_backend": jax.default_backend(), "x64": bool(jax.config.read("jax_enable_x64"))}
print(f"Run {RUN_ID} | Colab: {IN_COLAB}"); print(VERSIONS)

In [ ]:
# =====================================================================
# CONFIGURATION
# =====================================================================
DRIVE_MOUNT_POINT = Path("/content/drive"); DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
PROJECT_ROOT = Path(os.environ.get("RAR_PROJECT_ROOT", str(DRIVE_ROOT / "RAR_Universality_Research")))
if IN_KAGGLE and "RAR_PROJECT_ROOT" not in os.environ:          # optional: also runs unattended on Kaggle
    import shutil
    cands = [p.parent for p in Path("/kaggle/input").rglob("03_Processed_Data") if p.is_dir()]
    full = [c for c in cands if any((c / "05_Results" / "Posterior_Samples").glob("03_bayesian_full_modelC_posterior*.nc"))]
    if not (full or cands): raise FileNotFoundError("Add your RAR_Universality_Research dataset as a Kaggle input.")
    src = sorted(full or cands, key=lambda p: len(p.parts))[0]
    PROJECT_ROOT = Path("/kaggle/working/RAR_Universality_Research")
    if not PROJECT_ROOT.exists(): shutil.copytree(src, PROJECT_ROOT, ignore=shutil.ignore_patterns("01_Datasets"))
PROC = PROJECT_ROOT / "03_Processed_Data"
DIRS = {"processed": PROC, "models": PROJECT_ROOT / "04_Trained_Models", "tables": PROJECT_ROOT / "05_Results" / "Tables",
        "posterior": PROJECT_ROOT / "05_Results" / "Posterior_Samples", "logs": PROC / "logs"}           # read-only inputs
NB02 = {"points": PROC / "sparc_rar_points_baseline.csv", "galaxies": PROC / "sparc_rar_galaxies_baseline.csv", "npz": PROC / "sparc_rar_model_inputs.npz"}
AUDIT01 = PROC / "sparc_galaxy_audit.csv"            # Notebook 01: derived Mbar and gas fraction (documented fiducial definitions)
NB03_RUN_ID = os.environ.get("RAR_NB03_RUN_ID", "")
OUT_ROOT = PROJECT_ROOT / "05_Results" / "Injection_Recovery_Extended" / RUN_ID # new, versioned; never overwrites
NB06_DIR = PROJECT_ROOT / "05_Results" / "Injection_Recovery"                      # read-only: Notebook 06 A1 null distributions
NB04_DIR = PROJECT_ROOT / "05_Results" / "Diagnostics" / "04_GP_Debugged"          # read-only: full Model D posterior
NB05_DIR = PROJECT_ROOT / "05_Results" / "Population_Dependence"                   # read-only: real slopes
OUT = {k: OUT_ROOT / k for k in ("posterior", "tables", "figures", "diagnostics", "config", "logs")}

KPC_IN_M, KMS_IN_MS, LN10 = 3.0856775814913673e19, 1.0e3, float(np.log(10.0))
REF = {"a0": 1.20e-10, "beta": 0.5, "upsilon_disk": 0.5, "upsilon_bulge": 0.7}
UPS_SIGMA_DEX = {"D": 0.10, "E": 0.20}             # Model C uses the 0.10 dex width (verified against Notebook 03)

# ---------------- Population model (proposal Sec. 4.5) ----------------
COVARIATES = {   # name: (definition, source)
    "logMbar": ("log10 baryonic mass [Msun] = log10(1.33 M_HI + 0.5 L[3.6]); Notebook 01 fiducial definition", "sparc_galaxy_audit.csv"),
    "logSB0":  ("log10 central disk surface brightness at 3.6 um [Lsun/pc^2] (SPARC SBdisk)", "sparc_rar_galaxies_baseline.csv"),
    "fgas":    ("gas fraction 1.33 M_HI / Mbar (Notebook 01 fiducial definition)", "sparc_galaxy_audit.csv"),
}
POP_PARAMS = ["log10_a0", "beta", "log_sigma_int", "log_tau"]
# Slope priors (per 1 SD of the covariate). They are not specified by the proposal: weakly informative regularisation choices,
# with a x2 sensitivity test. The intercepts (values at the sample-mean property) keep Notebook 03's priors exactly.
SLOPE_PRIOR_SD = {"log10_a0": 0.25,      # dex of a0 per SD
                  "beta": 0.10,          # beta per SD
                  "log_sigma_int": 0.5,  # natural-log units per SD (x1.65 per SD at 1 sigma)
                  "log_tau": 0.5}
BETA_FLOOR = 0.01                         # beta_i <= floor is outside the RAR's domain (zero likelihood); reported if ever approached
FITS_ONE_AT_A_TIME = list(COVARIATES)     # primary analysis
RUN_JOINT_FIT = True                      # secondary analysis (all three covariates together)
# ---------------- Notebook 08 plan ----------------
N_SHUF = int(os.environ.get("RAR_N_SHUF", 20))               # C1: shuffled-amplitude null mocks (x 3 one-at-a-time fits)
C2_AMPLITUDES = [1.0, 2.0]                                   # C2: multiples of the Notebook 05 slopes, injected on correlated noise
N_C2 = int(os.environ.get("RAR_N_C2", 2))                    # C2: mocks per covariate and amplitude (1 fit each)
MOCK_SEED = 20261029                                         # different from Notebook 06 (20261028): new, independent mocks
MOCK_FIT = {"chains": int(os.environ.get("RAR_MOCK_CHAINS", 2)), "warmup": int(os.environ.get("RAR_MOCK_WARMUP", 500)),
            "samples": int(os.environ.get("RAR_MOCK_SAMPLES", 750))}
MOCK_RHAT_MAX, MOCK_ESS_MIN = float(os.environ.get("RAR_MOCK_RHAT_MAX", 1.05)), float(os.environ.get("RAR_MOCK_ESS_MIN", 100))   # env override for testing only
RESUME_FROM = os.environ.get("RAR_RESUME_FROM", "")          # path of an earlier 08 run folder to resume from ("" = fresh)
RUN_PRIOR_SENSITIVITY = False
FAMILY_ALPHA = 0.05                       # family-wise error rate over the declared test family (12 slopes)

# ---------------- Sampler (Notebook 03 settings; Model C converged with them) ----------------
SEED = 20261027
TARGET_ACCEPT, MAX_TREE_DEPTH = 0.90, 10
FIT = {"chains": int(os.environ.get("RAR_CHAINS", 4)), "warmup": int(os.environ.get("RAR_WARMUP", 1000)), "samples": int(os.environ.get("RAR_SAMPLES", 1500))}
RHAT_MAX, ESS_BULK_MIN, ESS_TAIL_MIN, DIVERGENCES_MAX, BFMI_MIN = 1.01, 400, 400, 0, 0.3
ILOO = {"post_draws": int(os.environ.get("RAR_ILOO_DRAWS", 300)), "mc": int(os.environ.get("RAR_ILOO_MC", 64))}
ALLOW_OVERWRITE = False
PROGRESS = not IN_KAGGLE
print("Project root:", PROJECT_ROOT); print("Outputs ->", OUT_ROOT)

## 3. Drive, Logging and Notebook 03 Inputs

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_ROOT}")
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()):
    raise FileExistsError(f"{OUT_ROOT} already exists and is not empty; refusing to overwrite")
for d in OUT.values():
    d.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUT["logs"] / f"08_extended_{RUN_ID}.log"
logging.raiseExceptions = False   # a Drive hiccup must not flood the output (seen in Notebook 06)
logger = logging.getLogger("rar08"); logger.setLevel(logging.INFO); logger.handlers.clear(); logger.propagate = False
_fh = logging.FileHandler(LOG_PATH, encoding="utf-8"); _fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
_sh = logging.StreamHandler(sys.stdout); _sh.setFormatter(logging.Formatter("%(levelname)s: %(message)s"))
logger.addHandler(_fh); logger.addHandler(_sh)

CHECKS = []
def record_check(section, name, passed, detail="", severity="critical"):
    status = "PASS" if passed else ("FAIL" if severity == "critical" else "WARN")
    CHECKS.append({"section": section, "check": name, "status": status, "severity": severity, "detail": str(detail)})
    (logger.info if passed else (logger.error if severity == "critical" else logger.warning))(f"[{status}] {section} | {name} | {detail}")
    if status == "FAIL":
        raise RuntimeError(f"Critical check failed: {section} | {name} | {detail}")
    return bool(passed)

def safe_path(path: Path) -> Path:
    if path.exists() and not ALLOW_OVERWRITE:
        new = path.with_name(f"{path.stem}_{RUN_ID}{path.suffix}"); logger.info(f"{path.name} exists; writing {new.name}"); return new
    return path

def sha256_of(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()
print("Log:", LOG_PATH)

In [ ]:
cfgs = []
for p in sorted(DIRS["models"].glob("03_bayesian_sampler_config*.json")):
    try:
        c = json.loads(p.read_text()); cm = c.get("full", {}).get("completed_models", {})
        cfgs.append({"file": p.name, "run_id": c.get("run_id"), "created": c.get("created_utc"), "full_models": ",".join(cm), "cfg": c})
    except Exception as exc:
        logger.warning(f"Could not read {p.name}: {exc}")
cfg_table = pd.DataFrame(cfgs)
display(cfg_table.drop(columns="cfg") if len(cfg_table) else "no Notebook 03 configurations found")
cands = [c for c in cfgs if "C" in c["full_models"].split(",") and (not NB03_RUN_ID or c["run_id"] == NB03_RUN_ID)]
record_check("NB03", "a Notebook 03 run with a completed full Model C exists", bool(cands), f"{len(cands)} candidate runs")
nb03 = max(cands, key=lambda c: c["created"]); CFG03 = nb03["cfg"]; RUN03 = nb03["run_id"]
print(f"Using Notebook 03 run {RUN03} (config {nb03['file']})")

def run_file(folder, stem, ext):
    # File written by run RUN03: suffixed name if present, else the base name.
    p1 = folder / f"{stem}_{RUN03}{ext}"; p0 = folder / f"{stem}{ext}"
    return p1 if p1.is_file() else p0

NC03 = {}
for m, info in CFG03["full"]["completed_models"].items():
    p = Path(info["netcdf"])
    if not p.is_file():                                         # stored path may be absolute from another mount; fall back to folder
        p = DIRS["posterior"] / Path(info["netcdf"]).name
    record_check("NB03", f"Model {m} posterior file exists", p.is_file(), str(p))
    attrs = xr.open_dataset(p, group="posterior", engine="h5netcdf").attrs
    record_check("NB03", f"Model {m} posterior carries run ID {RUN03}", attrs.get("run_id") == RUN03, f"file run_id {attrs.get('run_id')}")
    st = info["stats"]
    record_check("NB03", f"Model {m} met all convergence criteria in Notebook 03", bool(st["converged_by_all_criteria"]),
                 f"max R-hat {st['max_r_hat_all']:.4f}, min ESS {st['min_ess_bulk_all']:.0f}, divergences {st['divergences']}",
                 severity="warning" if os.environ.get("RAR_TEST_ALLOW_UNCONVERGED_NB03") == "1" else "critical")   # override: local testing only
    NC03[m] = p
chk03_path = run_file(DIRS["processed"], "03_bayesian_integrity_checks", ".csv")
chk03 = pd.read_csv(chk03_path)
record_check("NB03", "Notebook 03 check register has no FAIL rows", int((chk03.status == "FAIL").sum()) == 0,
             f"{chk03_path.name}: {len(chk03)} checks, {int((chk03.status=='FAIL').sum())} FAIL, {int((chk03.status=='WARN').sum())} WARN")
print("Notebook 03 warnings:"); display(chk03[chk03.status == "WARN"][["section", "check", "detail"]])
comp03_path = run_file(DIRS["tables"], "03_bayesian_model_comparison", ".csv")
comp03 = pd.read_csv(comp03_path) if comp03_path.is_file() else pd.DataFrame()
if len(comp03):
    display(comp03[[c for c in comp03.columns if c in ("stage", "model", "elpd_loo_galaxy", "p_loo", "n_pareto_k_gt_0.7", "sigma_int_median", "tau_median")]])
    bad_k = comp03[(comp03.get("n_pareto_k_gt_0.7", 0) > 0.1 * comp03.get("n_galaxies", 1))]
    record_check("NB03", "Notebook 03 galaxy-level PSIS-LOO reliable (Pareto k > 0.7 in < 10% of galaxies)", bad_k.empty,
                 bad_k[["stage", "model", "n_pareto_k_gt_0.7"]].values.tolist() if not bad_k.empty else "ok", severity="warning")

# Shared priors must be identical to Notebook 03
P03 = CFG03["priors"]
same_priors = (P03["log10_a0"]["low"] == -11.5 and P03["log10_a0"]["high"] == -8.5 and P03["beta"]["low"] == 0.05 and P03["beta"]["high"] == 1.0
               and P03["sigma_int"]["scale"] == 0.1 and P03["tau"]["scale"] == 0.1 and P03["upsilon_disk"]["sigma_dex"] == UPS_SIGMA_DEX["D"]
               and P03["upsilon_disk"]["median"] == REF["upsilon_disk"] and P03["upsilon_bulge"]["median"] == REF["upsilon_bulge"]
               and P03["inclination"]["low"] == 1.0 and P03["inclination"]["high"] == 90.0)
record_check("NB03", "shared priors in this notebook equal Notebook 03's saved priors", same_priors, "log10_a0, beta, sigma_int, tau, Upsilon, inclination bounds")
record_check("NB03", "Sigma_x convention unchanged (0: no documented baryonic-component errors)", CFG03.get("sigma_x_log_gbar", None) == 0.0, "0.0")

In [ ]:
for k in ("points", "galaxies", "npz"):
    rec = CFG03["input_checksums"].get(NB02[k].name)
    record_check("Sample", f"{NB02[k].name} unchanged since Notebook 03", rec is not None and sha256_of(NB02[k]) == rec, "SHA-256 compared")
pts = pd.read_csv(NB02["points"]); gals = pd.read_csv(NB02["galaxies"])
npz = np.load(NB02["npz"])
p_sorted = pts.sort_values(["galaxy_index", "point_index"]).reset_index(drop=True)
gtab = gals.sort_values("galaxy_index").reset_index(drop=True)
G = len(gtab); gi = p_sorted.galaxy_index.to_numpy(np.int32)
R0, V0, eV0 = npz["R_kpc"], npz["Vobs_kms"], npz["errV_kms"]
Vg0, Vd0, Vb0 = npz["Vgas_kms"], npz["Vdisk_ups1_kms"], npz["Vbul_ups1_kms"]
D0, eD0 = gtab.distance_mpc.to_numpy(float), gtab.e_distance_mpc.to_numpy(float)
I0, eI0 = gtab.inc_deg.to_numpy(float), gtab.e_inc_deg.to_numpy(float)
has_bulge = p_sorted.groupby("galaxy_index").Vbul.apply(lambda s: bool((s > 0).any())).reindex(range(G)).to_numpy()

# Positivity rule exactly as Notebook 03 (same seed and draw count); E's wider prior is evaluated for information only
def positivity(sig_dex, seed, n):
    rng = np.random.default_rng(seed)
    ud = 10 ** (np.log10(REF["upsilon_disk"]) + sig_dex * rng.standard_normal(n))
    ub = 10 ** (np.log10(REF["upsilon_bulge"]) + sig_dex * rng.standard_normal(n))
    p = np.ones(len(R0))
    for k in np.where(Vg0 <= 0)[0]:
        p[k] = float(np.mean(ud * Vd0[k] ** 2 + ub * Vb0[k] ** 2 + Vg0[k] * abs(Vg0[k]) > 0))
    return p
pos = CFG03["positivity_rule"]
p_pos = positivity(UPS_SIGMA_DEX["D"], CFG03["seed"], pos["mc_draws"])
gbar_ref = (Vg0 * np.abs(Vg0) + 0.5 * Vd0 ** 2 + 0.7 * Vb0 ** 2)
include = (p_pos >= pos["P_prior_min"]) & (gbar_ref > 0)
excl = sorted(zip(p_sorted.galaxy[~include], p_sorted.point_index[~include].astype(int)))
record_check("Sample", f"point set reproduces Notebook 03 ({pos['n_points_included']} points)", int(include.sum()) == pos["n_points_included"], f"{int(include.sum())}")
record_check("Sample", "excluded points identical to Notebook 03", [list(map(lambda v: v if isinstance(v, str) else int(v), e)) for e in excl] == pos["excluded_points"], excl)


## 4. Covariates, Likelihood and Model (identical to Notebooks 05 and 06)

In [ ]:
aud = pd.read_csv(AUDIT01)
cov = gtab[["galaxy", "galaxy_index", "SBdisk0_Lsun_pc2"]].merge(aud[["galaxy", "Mbar_1e9Msun_derived_fid", "fgas_derived_fid", "MHI_1e9Msun", "L36_1e9Lsun"]], on="galaxy", how="left")
cov["logMbar"] = np.log10(cov.Mbar_1e9Msun_derived_fid * 1e9)
cov["logSB0"] = np.log10(cov.SBdisk0_Lsun_pc2)
cov["fgas"] = cov.fgas_derived_fid
recheck = np.allclose(cov.Mbar_1e9Msun_derived_fid, 1.33 * cov.MHI_1e9Msun + 0.5 * cov.L36_1e9Lsun, rtol=1e-6)
record_check("Covariates", "Mbar reproduces 1.33 M_HI + 0.5 L[3.6] (Notebook 01 definition)", recheck, "")
record_check("Covariates", "galaxy order equals the model-input order", list(cov.galaxy_index) == list(range(G)), f"{G} galaxies")
for c in COVARIATES:
    record_check("Covariates", f"{c} finite for all {G} galaxies", bool(np.isfinite(cov[c]).all()), f"{int((~np.isfinite(cov[c])).sum())} non-finite")
COV_MEAN = {c: float(cov[c].mean()) for c in COVARIATES}; COV_SD = {c: float(cov[c].std(ddof=0)) for c in COVARIATES}
Zall = np.column_stack([(cov[c] - COV_MEAN[c]) / COV_SD[c] for c in COVARIATES])
cov_corr = pd.DataFrame(np.corrcoef(Zall.T), index=list(COVARIATES), columns=list(COVARIATES))
vif = {c: float(np.linalg.inv(cov_corr.values)[k, k]) for k, c in enumerate(COVARIATES)}
print("Covariate summary (physical units):"); display(cov[list(COVARIATES)].describe().T)
print("Correlation matrix of the standardised covariates:"); display(cov_corr.round(3))
print("Variance inflation factors (joint fit):", {k: round(v, 2) for k, v in vif.items()})
if max(vif.values()) > 2.5:
    logger.warning("Covariates are strongly correlated: joint-fit slopes will be partly degenerate. The one-at-a-time fits are primary (declared).")
cov_table = cov[["galaxy", "logMbar", "logSB0", "fgas"]].assign(**{f"z_{c}": Zall[:, k] for k, c in enumerate(COVARIATES)})

In [ ]:
def accelerations(D, inc_deg, ups_d, ups_b, idx, R0, V0, eV0, Vg0, Vd0, Vb0, D0, I0, xp=jnp):
    sD = (D / D0)[idx]
    s_inc = (xp.sin(xp.deg2rad(I0)) / xp.sin(xp.deg2rad(inc_deg)))[idx]
    R_m = R0 * sD * KPC_IN_M
    Vobs, eV = V0 * s_inc, eV0 * s_inc
    vbar2 = (Vg0 * xp.abs(Vg0) + ups_d[idx] * Vd0 ** 2 + ups_b[idx] * Vb0 ** 2) * sD
    g_obs = (Vobs * KMS_IN_MS) ** 2 / R_m
    g_bar = vbar2 * KMS_IN_MS ** 2 / R_m
    sig_V = 2.0 * eV / (Vobs * LN10)
    return g_obs, g_bar, sig_V

def f_rar(x, log10_a0, beta):
    t = jnp.exp(beta * (x - log10_a0) * LN10)
    return x - jnp.log10(-jnp.expm1(-t))

def fprime_rar(x, log10_a0, beta):
    t = jnp.exp(beta * (x - log10_a0) * LN10)
    return 1.0 - beta * t * jnp.exp(-t) / (-jnp.expm1(-t))

def residual_terms(params, data, nuis):
    log10_a0, beta = params[0], params[1]
    D, inc, ud, ub = nuis
    g_obs, g_bar, sig_V = accelerations(D, inc, ud, ub, data["idx"], data["R0"], data["V0"], data["eV0"],
                                        data["Vg0"], data["Vd0"], data["Vb0"], data["D0"], data["I0"])
    pos = g_bar > 0
    x = jnp.log10(jnp.where(pos, g_bar, 1e-10)); y = jnp.log10(g_obs)
    mu = f_rar(x, log10_a0, beta)
    return y - mu, sig_V, x, y, mu, pos

def galaxy_loglik_C(params, data, nuis):
    # Notebook 03 Model C likelihood (Sherman-Morrison, delta_i marginalised).
    log10_a0, beta, sig_int, tau, s = params
    r, sig_V, x, _, _, pos = residual_terms(params, data, nuis)
    d = (s * sig_V) ** 2 + (fprime_rar(x, log10_a0, beta) * data["sigx"]) ** 2 + sig_int ** 2
    seg = lambda v: jax.ops.segment_sum(v, data["idx"], num_segments=data["G"])
    S, A, Q, L, n = seg(1 / d), seg(r / d), seg(r ** 2 / d), seg(jnp.log(d)), seg(jnp.ones_like(d))
    ll = -0.5 * (Q - tau ** 2 * A ** 2 / (1 + tau ** 2 * S) + L + jnp.log1p(tau ** 2 * S) + n * jnp.log(2 * jnp.pi))
    return ll, jnp.all(pos)

def build_data_C(mask_points):
    rows = np.where(np.asarray(mask_points, bool))[0]
    gal_ids = np.unique(gi[rows]); remap = -np.ones(G, int); remap[gal_ids] = np.arange(len(gal_ids))
    idx = remap[gi[rows]]; Gs = len(gal_ids); bulge_local = np.where(has_bulge[gal_ids])[0]
    return {"idx": jnp.asarray(idx), "G": Gs, "gal_ids": gal_ids, "rows": rows, "bulge_local": jnp.asarray(bulge_local), "n_bulge": len(bulge_local),
            **{k: jnp.asarray(v[rows]) for k, v in dict(R0=R0, V0=V0, eV0=eV0, Vg0=Vg0, Vd0=Vd0, Vb0=Vb0).items()},
            "D0": jnp.asarray(D0[gal_ids]), "eD0": jnp.asarray(eD0[gal_ids]), "I0": jnp.asarray(I0[gal_ids]), "eI0": jnp.asarray(eI0[gal_ids]),
            "sigx": jnp.zeros(len(rows))}

def galaxy_loglik_pop(a_g, b_g, s_g, t_g, data, nuis):
    # Model C likelihood with per-galaxy log10 a0, beta, sigma_int, tau (arrays of length G).
    D, inc, ud, ub = nuis; idx = data["idx"]
    g_obs, g_bar, sig_V = accelerations(D, inc, ud, ub, idx, data["R0"], data["V0"], data["eV0"], data["Vg0"], data["Vd0"], data["Vb0"], data["D0"], data["I0"])
    pos = g_bar > 0
    x = jnp.log10(jnp.where(pos, g_bar, 1e-10)); y = jnp.log10(g_obs)
    a_p, b_p = a_g[idx], b_g[idx]
    r = y - f_rar(x, a_p, b_p)
    d = sig_V ** 2 + (fprime_rar(x, a_p, b_p) * data["sigx"]) ** 2 + s_g[idx] ** 2
    seg = lambda v: jax.ops.segment_sum(v, idx, num_segments=data["G"])
    S, A, Q, L, n = seg(1 / d), seg(r / d), seg(r ** 2 / d), seg(jnp.log(d)), seg(jnp.ones_like(d))
    ll = -0.5 * (Q - t_g ** 2 * A ** 2 / (1 + t_g ** 2 * S) + L + jnp.log1p(t_g ** 2 * S) + n * jnp.log(2 * jnp.pi))
    return ll, jnp.all(pos)

include_mask = include
data_full = build_data_C(include_mask)
Z_full = jnp.asarray(Zall[data_full["gal_ids"]])
print(f"Data: {data_full['G']} galaxies, {len(data_full['rows'])} points")

In [ ]:
ds_post = xr.open_dataset(NC03["C"], group="posterior", engine="h5netcdf")
ds_ll = xr.open_dataset(NC03["C"], group="log_likelihood", engine="h5netcdf")
names_nc = [str(x) for x in ds_post.galaxy.values]; names_here = list(gtab.galaxy.to_numpy()[data_full["gal_ids"]])
record_check("Regression", "Notebook 03 Model C galaxy order equals this notebook's", names_nc == names_here, f"{len(names_nc)} galaxies")
bulge_pos = [names_here.index(str(n)) for n in ds_post.bulge_galaxy.values] if "bulge_galaxy" in ds_post.dims else []
def nuis_from_nc(ds, c, d):
    ub = np.full(data_full["G"], REF["upsilon_bulge"])
    if bulge_pos: ub[bulge_pos] = ds["ups_bulge"].values[c, d]
    return (jnp.asarray(ds["D"].values[c, d]), jnp.asarray(ds["inc"].values[c, d]), jnp.asarray(ds["ups_disk"].values[c, d]), jnp.asarray(ub))
rng = np.random.default_rng(SEED); nchain, ndraw = ds_post.sizes["chain"], ds_post.sizes["draw"]; dev_saved, dev_C = 0.0, 0.0
for _ in range(10):
    c, d = int(rng.integers(nchain)), int(rng.integers(ndraw))
    th = [float(ds_post[k].values[c, d]) for k in ("log10_a0", "beta", "sigma_int", "tau")]
    nuis = nuis_from_nc(ds_post, c, d); Gn = data_full["G"]
    ll_p, _ = galaxy_loglik_pop(*(jnp.full(Gn, v) for v in th), data_full, nuis)
    ll_c, _ = galaxy_loglik_C((*th, 1.0), data_full, nuis)
    dev_saved = max(dev_saved, float(np.max(np.abs(np.asarray(ll_p) - ds_ll["galaxy_loglik"].values[c, d]))))
    dev_C = max(dev_C, float(np.max(np.abs(np.asarray(ll_p) - np.asarray(ll_c)))))
record_check("Regression", "zero-slope likelihood = Notebook 03 saved Model C log-likelihoods (10 draws, all galaxies)", dev_saved < 1e-6, f"max |diff| = {dev_saved:.2e}")
record_check("Regression", "zero-slope likelihood = Notebook 03 galaxy_loglik_C", dev_C < 1e-8, f"max |diff| = {dev_C:.2e}")
REGRESSION = {"max_abs_diff_vs_saved_NB03": dev_saved, "max_abs_diff_vs_C": dev_C}

In [ ]:
PRIORS_SHARED = {"log10_a0": P03["log10_a0"], "beta": P03["beta"], "sigma_int": P03["sigma_int"], "tau": P03["tau"]}
def pop_model(data, cfg):
    Z = cfg["Z"]; K = Z.shape[1]; mult = cfg.get("slope_prior_mult", 1.0)
    log10_a0 = numpyro.sample("log10_a0", dist.Uniform(P03["log10_a0"]["low"], P03["log10_a0"]["high"]))
    beta = numpyro.sample("beta", dist.Uniform(P03["beta"]["low"], P03["beta"]["high"]))
    sig_int = numpyro.sample("sigma_int", dist.HalfNormal(P03["sigma_int"]["scale"]))
    tau = numpyro.sample("tau", dist.HalfNormal(P03["tau"]["scale"]))
    if K:
        s = {p: numpyro.sample(f"slope_{p}", dist.Normal(jnp.zeros(K), mult * SLOPE_PRIOR_SD[p]).to_event(1)) for p in POP_PARAMS}
        a_g = log10_a0 + Z @ s["log10_a0"]; b_g = beta + Z @ s["beta"]
        s_g = sig_int * jnp.exp(Z @ s["log_sigma_int"]); t_g = tau * jnp.exp(Z @ s["log_tau"])
    else:
        Gn = data["G"]; a_g, b_g, s_g, t_g = (jnp.full(Gn, log10_a0), jnp.full(Gn, beta), jnp.full(Gn, sig_int), jnp.full(Gn, tau))
    with numpyro.plate("galaxies", data["G"]):                       # Notebook 03 Model C nuisance sampling, verbatim
        zD = numpyro.sample("zD", dist.Normal(0, 1))
        inc = numpyro.sample("inc", dist.TruncatedNormal(data["I0"], data["eI0"], low=P03["inclination"]["low"], high=P03["inclination"]["high"]))
        zUd = numpyro.sample("zUd", dist.Normal(0, 1))
    D = numpyro.deterministic("D", data["D0"] * jnp.exp(zD * data["eD0"] / data["D0"]))
    ud = numpyro.deterministic("ups_disk", P03["upsilon_disk"]["median"] * 10 ** (P03["upsilon_disk"]["sigma_dex"] * zUd))
    ub = jnp.full(data["G"], REF["upsilon_bulge"])
    if data["n_bulge"] > 0:
        with numpyro.plate("bulge_galaxies", data["n_bulge"]):
            zUb = numpyro.sample("zUb", dist.Normal(0, 1))
        ub = ub.at[data["bulge_local"]].set(P03["upsilon_bulge"]["median"] * 10 ** (P03["upsilon_bulge"]["sigma_dex"] * zUb))
        numpyro.deterministic("ups_bulge", ub[data["bulge_local"]])
    numpyro.deterministic("beta_min_over_galaxies", jnp.min(b_g))
    ll, ok = galaxy_loglik_pop(a_g, b_g, s_g, t_g, data, (D, inc, ud, ub))
    numpyro.factor("loglik", jnp.where(ok & (jnp.min(b_g) > BETA_FLOOR), jnp.sum(ll), -jnp.inf))

def fit_cfg(name, covs, mult=1.0):
    k = [list(COVARIATES).index(c) for c in covs]
    return {"name": name, "covariates": list(covs), "Z": Z_full[:, k] if k else jnp.zeros((data_full["G"], 0)), "slope_prior_mult": mult}

C_MED = {k: float(np.median(ds_post[k].values)) for k in ("log10_a0", "beta", "sigma_int", "tau")}
def init_values(cfg):
    v = {**C_MED, "zD": jnp.zeros(data_full["G"]), "inc": jnp.clip(data_full["I0"], 2.0, 89.5), "zUd": jnp.zeros(data_full["G"])}
    if data_full["n_bulge"]: v["zUb"] = jnp.zeros(data_full["n_bulge"])
    for p in POP_PARAMS:
        if cfg["Z"].shape[1]: v[f"slope_{p}"] = jnp.zeros(cfg["Z"].shape[1])
    return v

from numpyro.infer.util import log_density
lp0, _ = log_density(pop_model, (data_full, fit_cfg("baseline", [])), {}, init_values(fit_cfg("baseline", [])))
record_check("Model", "baseline (no slopes) log density finite at the initial values", bool(np.isfinite(float(lp0))), f"{float(lp0):.2f}")
cfg_test = fit_cfg("joint", list(COVARIATES)); iv = init_values(cfg_test)
lpj, trj = log_density(pop_model, (data_full, cfg_test), {}, iv)
lpb, trb = log_density(pop_model, (data_full, fit_cfg("baseline", [])), {}, init_values(fit_cfg("baseline", [])))
record_check("Model", "zero slopes give the baseline likelihood exactly", abs(float(trj["loglik"]["fn"].log_factor) - float(trb["loglik"]["fn"].log_factor)) < 1e-9, "joint vs baseline loglik factor")

In [ ]:
def run_fit(cfg, seed):
    kernel = NUTS(pop_model, target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH, init_strategy=init_to_value(values=init_values(cfg)))
    method = "parallel" if len(jax.devices()) >= FIT["chains"] else "sequential"
    mcmc = MCMC(kernel, num_warmup=FIT["warmup"], num_samples=FIT["samples"], num_chains=FIT["chains"], chain_method=method, progress_bar=PROGRESS)
    t0 = time.time()
    mcmc.run(jax.random.PRNGKey(seed), data_full, cfg, extra_fields=("diverging", "num_steps", "energy", "accept_prob"))
    jax.block_until_ready(mcmc.get_samples())
    return mcmc, time.time() - t0

def _rank_norm(x):
    r = sst.rankdata(x.ravel()).reshape(x.shape); return sst.norm.ppf((r - 3 / 8) / (x.size + 1 / 4))
def rank_rhat(x):
    return max(float(split_gelman_rubin(_rank_norm(x)[..., None])[0]), float(split_gelman_rubin(_rank_norm(np.abs(x - np.median(x)))[..., None])[0]))
def bulk_ess(x): return float(effective_sample_size(_rank_norm(x)[..., None])[0])
def tail_ess(x):
    q05, q95 = np.quantile(x, [0.05, 0.95])
    return float(min(effective_sample_size((x <= q05).astype(float)[..., None])[0], effective_sample_size((x >= q95).astype(float)[..., None])[0]))
SAMPLED = ["log10_a0", "beta", "sigma_int", "tau", "zD", "inc", "zUd", "zUb"] + [f"slope_{p}" for p in POP_PARAMS]
def diagnostics(mcmc):
    smp = mcmc.get_samples(group_by_chain=True); rows = []
    for k in SAMPLED:
        if k not in smp: continue
        v = np.asarray(smp[k]); flat = v.reshape(v.shape[0], v.shape[1], -1)
        rh = [rank_rhat(flat[..., j]) for j in range(flat.shape[2])]; eb = [bulk_ess(flat[..., j]) for j in range(flat.shape[2])]
        et = [tail_ess(flat[..., j]) for j in range(flat.shape[2])]
        rows.append({"parameter": k, "n_elements": flat.shape[2], "max_r_hat": max(rh), "min_ess_bulk": min(eb), "min_ess_tail": min(et)})
    dg = pd.DataFrame(rows); ex = mcmc.get_extra_fields(group_by_chain=True)
    div = np.asarray(ex["diverging"]); steps = np.asarray(ex["num_steps"]); en = np.asarray(ex["energy"])
    st = {"divergences": int(div.sum()), "divergences_per_chain": div.sum(1).tolist(), "max_tree_depth_hits": int((steps >= 2 ** MAX_TREE_DEPTH - 1).sum()),
          "mean_accept_prob": float(np.asarray(ex["accept_prob"]).mean()), "bfmi_per_chain": [float(np.mean(np.diff(e) ** 2) / np.var(e)) for e in en],
          "max_r_hat_all": float(dg.max_r_hat.max()), "min_ess_bulk_all": float(dg.min_ess_bulk.min()), "min_ess_tail_all": float(dg.min_ess_tail.min())}
    st["converged_by_all_criteria"] = bool(st["max_r_hat_all"] < RHAT_MAX and st["min_ess_bulk_all"] >= ESS_BULK_MIN and st["min_ess_tail_all"] >= ESS_TAIL_MIN
                                           and st["divergences"] <= DIVERGENCES_MAX and st["max_tree_depth_hits"] == 0 and min(st["bfmi_per_chain"]) >= BFMI_MIN)
    return dg, st

def save_fit(mcmc, cfg, path):
    smp = mcmc.get_samples(group_by_chain=True); ex = mcmc.get_extra_fields(group_by_chain=True)
    C_, S_ = np.asarray(smp["log10_a0"]).shape; names = gtab.galaxy.to_numpy()[data_full["gal_ids"]].astype(str)
    post, coords = {}, {"chain": np.arange(C_), "draw": np.arange(S_), "galaxy": names, "covariate": cfg["covariates"]}
    if data_full["n_bulge"]: coords["bulge_galaxy"] = names[np.asarray(data_full["bulge_local"])]
    for k, v in smp.items():
        v = np.asarray(v)
        if v.ndim == 2: post[k] = (("chain", "draw"), v)
        elif k.startswith("slope_"): post[k] = (("chain", "draw", "covariate"), v)
        elif v.shape[2] == data_full["G"]: post[k] = (("chain", "draw", "galaxy"), v)
        elif v.shape[2] == data_full["n_bulge"]: post[k] = (("chain", "draw", "bulge_galaxy"), v)
    if not cfg["covariates"]: coords.pop("covariate")
    path = safe_path(path)
    attrs = {"run_id": RUN_ID, "fit": cfg["name"], "covariates": json.dumps(cfg["covariates"]), "slope_prior_mult": cfg["slope_prior_mult"],
             "covariate_mean": json.dumps(COV_MEAN), "covariate_sd": json.dumps(COV_SD), "notebook03_run": RUN03}
    xr.Dataset(post, coords=coords, attrs=attrs).to_netcdf(path, group="posterior", mode="w", engine="h5netcdf")
    xr.Dataset({k2: (("chain", "draw"), np.asarray(ex[k1])) for k1, k2 in (("diverging", "diverging"), ("num_steps", "n_steps"), ("energy", "energy"), ("accept_prob", "acceptance_rate"))},
               coords={"chain": np.arange(C_), "draw": np.arange(S_)}).to_netcdf(path, group="sample_stats", mode="a", engine="h5netcdf")
    return path

## 5. Inputs from Notebooks 04, 05 and 06 (read-only)

In [ ]:
cands = sorted(NB04_DIR.glob("*/posterior/04d_gp_full_model_D_posterior*.nc"))
record_check("Inputs", "Notebook 04 full Model D posterior found", bool(cands), str(cands[-1]) if cands else f"none under {NB04_DIR}")
D_NC = cands[-1]
dpost = xr.open_dataset(D_NC, group="posterior", engine="h5netcdf")
names_D = [str(n) for n in dpost.galaxy.values]
record_check("Inputs", "Model D galaxy order equals this notebook's", names_D == names_here, f"{len(names_D)} galaxies")
TRUTH_CORR = {k: float(np.median(dpost[k].values)) for k in ("log10_a0", "beta", "tau", "sigma_int")}
A_TRUE = np.median(dpost["A"].values.reshape(-1, len(names_D)), axis=0); ELL_TRUE = np.median(dpost["ell"].values.reshape(-1, len(names_D)), axis=0)
TRUTH_WHITE = {k: float(np.median(ds_post[k].values)) for k in ("log10_a0", "beta", "tau", "sigma_int")}
print("A1 truth (Model D medians):", {k: round(v, 4) for k, v in TRUTH_CORR.items()}, "| A_i median", round(float(np.median(A_TRUE)), 2), "km/s; l_i median", round(float(np.median(ELL_TRUE)), 2), "kpc")
print("A2/B truth (NB03 Model C medians):", {k: round(v, 4) for k, v in TRUTH_WHITE.items()})
corrA = {c: float(np.corrcoef(np.log(A_TRUE), Zall[data_full["gal_ids"], k])[0, 1]) for k, c in enumerate(COVARIATES)}
print("Correlation of ln A_i (Model D) with the covariates:", {k: round(v, 2) for k, v in corrA.items()})

runs5 = sorted(NB05_DIR.glob("*/tables/05_slopes_all_fits.csv"))
record_check("Inputs", "Notebook 05 slope table found", bool(runs5), str(runs5[-1]) if runs5 else f"none under {NB05_DIR}")
slopes5 = pd.read_csv(runs5[-1]); real = slopes5[slopes5.fit.isin([f"one_{c}" for c in COVARIATES])].copy()
cfg5 = json.loads((runs5[-1].parents[1] / "config" / "05_population_config.json").read_text())
record_check("Inputs", "Notebook 05 used the same slope priors and covariate standardisation",
             cfg5["slope_prior_sd_per_SD"] == SLOPE_PRIOR_SD and all(abs(cfg5["covariate_mean"][c] - COV_MEAN[c]) < 1e-9 and abs(cfg5["covariate_sd"][c] - COV_SD[c]) < 1e-9 for c in COVARIATES),
             runs5[-1].parents[1].name)
REAL_SLOPE = {(r.parameter, r.covariate): (r.slope_per_SD_median, r.slope_per_SD_sd) for r in real.itertuples()}
display(real[["parameter", "covariate", "slope_per_SD_median", "slope_per_SD_sd", "verdict_family_wise"]])

# Notebook 06: A1 (correlated-noise) null distribution of every slope, used to calibrate the C2 power
runs6 = sorted(NB06_DIR.glob("*/tables/06_false_positive_rates.csv"))
record_check("Inputs", "Notebook 06 false-positive table found", bool(runs6), str(runs6[-1]) if runs6 else f"none under {NB06_DIR}")
fp6 = pd.read_csv(runs6[-1]); a1 = fp6[fp6.scenario == "A1_null_correlated"]
record_check("Inputs", "Notebook 06 A1 null has at least 10 usable mocks per slope", bool(len(a1) == 12 and (a1.n_mocks >= 10).all()) if len(a1) else False,
             a1[["parameter", "covariate", "n_mocks"]].to_dict("records") if len(a1) else "no A1 rows", severity="warning")
A1_NULL = {(r.parameter, r.covariate): (r.null_slope_mean, r.null_slope_sd, int(r.n_mocks)) for r in a1.itertuples()}
print("Notebook 06 run used for the A1 null:", runs6[-1].parents[1].name)

## 6. Mock Generator (Notebook 06, with an optional amplitude vector)

The only change: `make_mock` accepts `A_vec`, the per-galaxy correlated-noise amplitudes. `A_vec=None` reproduces Notebook 06 exactly (A_i from Model D). The generator checks from Notebook 06 are re-run.

In [ ]:
idx_np = np.asarray(data_full["idx"]); rows = data_full["rows"]; Gn = data_full["G"]
R0f, V0f, eV0f = R0[rows], V0[rows], eV0[rows]; Vg0f, Vd0f, Vb0f = Vg0[rows], Vd0[rows], Vb0[rows]
D0g, eD0g, I0g, eI0g = (np.asarray(data_full[k]) for k in ("D0", "eD0", "I0", "eI0"))
REL_ERR = eV0f / V0f                                   # fractional velocity error per point (kept in the mocks)
W_LOG = 2.0 / (V0f * LN10)                             # J in log-acceleration per km/s (catalogue velocities; as Notebook 04)
bulge_mask = np.zeros(Gn, bool); bulge_mask[np.asarray(data_full["bulge_local"])] = True
Zg = np.asarray(Z_full)
GAL_PTS = [np.where(idx_np == g)[0] for g in range(Gn)]

def f_rar_np(x, la0, b):
    t = np.exp(b * (x - la0) * LN10); return x - np.log10(-np.expm1(-t))

def theta_galaxy(truth, slopes, covs):
    a = np.full(Gn, truth["log10_a0"]); b = np.full(Gn, truth["beta"]); s = np.full(Gn, truth["sigma_int"]); t = np.full(Gn, truth["tau"])
    if slopes:
        for c in covs:
            z = Zg[:, list(COVARIATES).index(c)]
            a = a + slopes.get(("log10_a0", c), 0.0) * z; b = b + slopes.get(("beta", c), 0.0) * z
            s = s * np.exp(slopes.get(("log_sigma_int", c), 0.0) * z); t = t * np.exp(slopes.get(("log_tau", c), 0.0) * z)
    return a, b, s, t

def make_mock(rng, truth, gp=False, slopes=None, covs=(), A_vec=None):
    A_use = A_TRUE if A_vec is None else np.asarray(A_vec)
    D = D0g * np.exp(rng.standard_normal(Gn) * eD0g / D0g)
    inc = sst.truncnorm.rvs((1.0 - I0g) / eI0g, (90.0 - I0g) / eI0g, loc=I0g, scale=eI0g, random_state=rng)
    for _ in range(100):                                   # redraw Upsilon only where some included point would have g_bar <= 0
        ud = REF["upsilon_disk"] * 10 ** (UPS_SIGMA_DEX["D"] * rng.standard_normal(Gn))
        ub = REF["upsilon_bulge"] * 10 ** (UPS_SIGMA_DEX["D"] * rng.standard_normal(Gn))
        vb2 = Vg0f * np.abs(Vg0f) + ud[idx_np] * Vd0f ** 2 + ub[idx_np] * Vb0f ** 2
        bad = np.unique(idx_np[vb2 <= 0])
        if not len(bad): break
    _, g_bar, _ = accelerations(D, inc, ud, ub, idx_np, R0f, V0f, eV0f, Vg0f, Vd0f, Vb0f, D0g, I0g, xp=np)
    x = np.log10(g_bar)
    a_g, b_g, s_g, t_g = theta_galaxy(truth, slopes, covs)
    sigV = 2.0 * REL_ERR / LN10
    y = f_rar_np(x, a_g[idx_np], b_g[idx_np]) + rng.normal(0, t_g)[idx_np] + rng.normal(0, 1, len(x)) * np.sqrt(sigV ** 2 + s_g[idx_np] ** 2)
    if gp:
        for g, pts in enumerate(GAL_PTS):
            r = R0f[pts]; K = A_use[g] ** 2 * np.exp(-0.5 * ((r[:, None] - r[None, :]) / ELL_TRUE[g]) ** 2)
            C = W_LOG[pts][:, None] * W_LOG[pts][None, :] * K + 1e-12 * np.eye(len(pts))
            y[pts] += np.linalg.cholesky(C) @ rng.standard_normal(len(pts))
    # back to observed velocities at the catalogue distance and inclination (model's forward map inverted exactly)
    s_inc = (np.sin(np.deg2rad(I0g)) / np.sin(np.deg2rad(inc)))[idx_np]; sD = (D / D0g)[idx_np]
    V0m = np.sqrt(10 ** y * R0f * sD * KPC_IN_M) / KMS_IN_MS / s_inc
    mock = dict(data_full); mock["V0"] = jnp.asarray(V0m); mock["eV0"] = jnp.asarray(REL_ERR * V0m)
    return mock, {"D": D, "inc": inc, "ud": ud, "ub": ub, "y": y, "x": x}

# ---- validation of the generator
rng_v = np.random.default_rng(MOCK_SEED - 1)
mk, tr = make_mock(rng_v, TRUTH_CORR, gp=True)
go, gb, sv = accelerations(jnp.asarray(tr["D"]), jnp.asarray(tr["inc"]), jnp.asarray(tr["ud"]), jnp.asarray(tr["ub"]), mk["idx"], mk["R0"], mk["V0"], mk["eV0"],
                           mk["Vg0"], mk["Vd0"], mk["Vb0"], mk["D0"], mk["I0"])
record_check("Generator", "model forward map with the true nuisance returns the generated log g_obs", float(np.max(np.abs(np.log10(np.asarray(go)) - tr["y"]))) < 1e-10,
             f"max |diff| {float(np.max(np.abs(np.log10(np.asarray(go)) - tr['y']))):.2e}")
record_check("Generator", "model sigma_V equals the intended (catalogue fractional) error", float(np.max(np.abs(np.asarray(sv) - 2 * REL_ERR / LN10))) < 1e-12, "")
record_check("Generator", "model g_bar with the true nuisance equals the generator's", float(np.max(np.abs(np.log10(np.asarray(gb)) - tr["x"]))) < 1e-10, "")
# covariance check for the largest galaxy: empirical vs intended (2000 noise draws, no RAR mean)
g_big = int(np.argmax([len(p) for p in GAL_PTS])); pts = GAL_PTS[g_big]; r = R0f[pts]
C_int = (W_LOG[pts][:, None] * W_LOG[pts][None, :] * A_TRUE[g_big] ** 2 * np.exp(-0.5 * ((r[:, None] - r[None, :]) / ELL_TRUE[g_big]) ** 2)
         + np.diag((2 * REL_ERR[pts] / LN10) ** 2 + TRUTH_CORR["sigma_int"] ** 2) + TRUTH_CORR["tau"] ** 2)
Lc = np.linalg.cholesky(C_int + 1e-12 * np.eye(len(pts))); E = (Lc @ rng_v.standard_normal((len(pts), 4000))).T
rel = float(np.linalg.norm(np.cov(E.T) - C_int) / np.linalg.norm(C_int))
record_check("Generator", f"noise covariance reproduces Notebook 04's C_i (largest galaxy, n = {len(pts)})", rel < 0.1, f"relative Frobenius error {rel:.3f} (4000 draws)")
# ---- shuffled amplitudes: a permutation keeps the distribution of A_i but removes its link to the covariates
rng_s = np.random.default_rng(MOCK_SEED - 2); perm = rng_s.permutation(Gn)
record_check("Generator", "a shuffle is a permutation of the Model D amplitudes", bool(np.allclose(np.sort(A_TRUE[perm]), np.sort(A_TRUE))), "")
mk0, _ = make_mock(np.random.default_rng(5), TRUTH_CORR, gp=True); mk1, _ = make_mock(np.random.default_rng(5), TRUTH_CORR, gp=True, A_vec=A_TRUE)
record_check("Generator", "A_vec = Model D amplitudes reproduces Notebook 06's generator exactly", bool(np.array_equal(np.asarray(mk0["V0"]), np.asarray(mk1["V0"]))), "")
shuf_corr = []
for s in range(200):
    p_ = np.random.default_rng([MOCK_SEED, 99, s]).permutation(Gn)
    shuf_corr.append(np.corrcoef(np.log(A_TRUE[p_]), Zall[data_full["gal_ids"], 0])[0, 1])
print(f"corr(ln A, logMbar): Model D {corrA['logMbar']:+.2f}; after shuffling: mean {np.mean(shuf_corr):+.3f}, SD {np.std(shuf_corr):.3f} (200 permutations)")

## 7. Plan and Mock-Fit Machinery

In [ ]:
def run_fit_mock(data, cfg, seed):
    kernel = NUTS(pop_model, target_accept_prob=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH, init_strategy=init_to_value(values=init_values(cfg)))
    method = "parallel" if len(jax.devices()) >= MOCK_FIT["chains"] else "sequential"
    mcmc = MCMC(kernel, num_warmup=MOCK_FIT["warmup"], num_samples=MOCK_FIT["samples"], num_chains=MOCK_FIT["chains"], chain_method=method, progress_bar=False)
    t0 = time.time(); mcmc.run(jax.random.PRNGKey(seed), data, cfg, extra_fields=("diverging", "num_steps", "energy", "accept_prob"))
    jax.block_until_ready(mcmc.get_samples()); return mcmc, time.time() - t0

KEY_SITES = ["log10_a0", "beta", "sigma_int", "tau"] + [f"slope_{p}" for p in POP_PARAMS]
def mock_summary(mcmc, cfg, meta):
    smp_c = mcmc.get_samples(group_by_chain=True); smp = mcmc.get_samples(); ex = mcmc.get_extra_fields(group_by_chain=True)
    rh, eb = [], []
    for k in KEY_SITES:
        v = np.asarray(smp_c[k]); v = v.reshape(v.shape[0], v.shape[1], -1)
        for j in range(v.shape[2]): rh.append(rank_rhat(v[..., j])); eb.append(bulk_ess(v[..., j]))
    div = int(np.asarray(ex["diverging"]).sum()); steps = np.asarray(ex["num_steps"])
    usable = bool(max(rh) < MOCK_RHAT_MAX and min(eb) >= MOCK_ESS_MIN and div == 0)
    rows = []
    for p in POP_PARAMS:
        x = np.asarray(smp[f"slope_{p}"])[:, 0]; c = cfg["covariates"][0]
        q = np.quantile(x, [0.025, 0.975]); med, sd = float(np.median(x)), float(x.std())
        rows.append({**meta, "parameter": p, "covariate": c, "slope_median": med, "slope_sd": sd, "ci95_low": q[0], "ci95_high": q[1],
                     "z": med / sd, "supported": bool(abs(med / sd) > ZCRIT), "usable": usable, "max_rhat_key": max(rh), "min_ess_key": min(eb),
                     "divergences": div, "tree_depth_hits": int((steps >= 2 ** MAX_TREE_DEPTH - 1).sum()),
                     "intercept_log10_a0": float(np.median(np.asarray(smp["log10_a0"]))), "intercept_sigma_int": float(np.median(np.asarray(smp["sigma_int"]))),
                     "intercept_tau": float(np.median(np.asarray(smp["tau"])))})
    return rows

N_FAMILY = len(POP_PARAMS) * len(FITS_ONE_AT_A_TIME); ALPHA_P = FAMILY_ALPHA / N_FAMILY; ZCRIT = float(sst.norm.ppf(1 - ALPHA_P / 2))
PLAN = []                                      # (mock_id, scenario, shuffle?, injected-slopes dict, covariates of the injection, fits to run)
for m in range(N_SHUF): PLAN.append((f"C1_{m:03d}", "C1_null_shuffled", True, None, (), list(FITS_ONE_AT_A_TIME)))
for c in FITS_ONE_AT_A_TIME:
    for amp in C2_AMPLITUDES:
        for m in range(N_C2):
            sl = {(p, c): amp * REAL_SLOPE[(p, c)][0] for p in POP_PARAMS}
            PLAN.append((f"C2_{c}_{amp:.1f}_{m:02d}", f"C2_inject_corr_{amp:.1f}x", False, sl, (c,), [c]))
n_fits = sum(len(p[5]) for p in PLAN)
print(f"Plan: {len(PLAN)} mocks, {n_fits} fits (about {n_fits * 1.7 / 60:.1f} h on Colab at ~1.7 min per fit)")
RES_CSV = OUT["tables"] / "08_mock_fit_results.csv"
done = pd.DataFrame()
if RESUME_FROM:
    prev = Path(RESUME_FROM) / "tables" / "08_mock_fit_results.csv"
    record_check("Resume", "previous results file found", prev.is_file(), str(prev))
    done = pd.read_csv(prev); done.to_csv(RES_CSV, index=False)
    print(f"Resuming: {done.drop_duplicates(['mock_id', 'fit_covariate']).shape[0]} fits already done")

## 8. Run the Mocks

All mocks contain correlated noise and use the Model D truth. C1 mocks draw a fresh permutation of A_i from their own seed; C2 mocks use the unshuffled A_i (as Notebook 06's A1) plus the injected slopes. Results are appended after each fit.

In [ ]:
t_start = time.time(); n_done = 0
for k, (mid, scen, shuffle, sl, inj_covs, fits) in enumerate(PLAN):
    todo = [c for c in fits if done.empty or not ((done.mock_id == mid) & (done.fit_covariate == c)).any()]
    if not todo: continue
    rng = np.random.default_rng([MOCK_SEED, k])
    A_vec = A_TRUE[rng.permutation(Gn)] if shuffle else None
    mock, _ = make_mock(rng, TRUTH_CORR, gp=True, slopes=sl, covs=inj_covs, A_vec=A_vec)
    for c in todo:
        cfg = fit_cfg(f"one_{c}", [c])
        try:
            mc, secs = run_fit_mock(mock, cfg, MOCK_SEED + 1000 * k + list(COVARIATES).index(c))
            rows = mock_summary(mc, cfg, {"mock_id": mid, "scenario": scen, "fit_covariate": c, "injected_covariate": ",".join(inj_covs) or "-", "seconds": secs})
            for rr in rows:
                rr["injected_slope"] = (sl or {}).get((rr["parameter"], c), 0.0)
                if A_vec is not None: rr["corr_lnA_logMbar_this_mock"] = float(np.corrcoef(np.log(A_vec), Zall[data_full["gal_ids"], 0])[0, 1])
        except Exception as exc:
            logger.error(f"{mid} fit {c} FAILED: {exc}"); rows = [{"mock_id": mid, "scenario": scen, "fit_covariate": c, "error": str(exc)}]
        pd.DataFrame(rows).to_csv(RES_CSV, mode="a", header=not RES_CSV.exists(), index=False)
        n_done += 1
    logger.info(f"{mid} ({scen}) done; {n_done} fits this session; {(time.time() - t_start) / 3600:.2f} h elapsed")
res = pd.read_csv(RES_CSV)
if "error" in res: record_check("Mocks", "no mock fit raised an error", bool(res["error"].isna().all()), f"{int(res['error'].notna().sum())} errors", severity="warning")
res = res[res["parameter"].notna()] if "parameter" in res else res
use = res[res.usable == True]
summary_use = res.groupby("scenario").agg(fits=("mock_id", lambda s: s.size // 4), usable_fraction=("usable", "mean")).reset_index()
display(summary_use)
record_check("Mocks", "at least 90% of mock fits usable", bool((summary_use.usable_fraction >= 0.9).all()), summary_use.to_dict("records"), severity="warning")

## 9. C1 — Shuffled-Amplitude Null

For each slope: the false-positive rate under the naive Bonferroni rule, the null mean and SD, and the real Notebook 05 slope calibrated against this null,
z_cal = (real − null mean) / √(null SD² + real posterior SD²), exactly as in Notebook 06. Because the null SD is estimated from N mocks, the matching cutoff is the Student-t quantile with N − 1 degrees of freedom at the Bonferroni level, and both cutoffs are reported. The C1 null mean and SD are compared with Notebook 06's A1 null: if they agree, the A1 calibration was not driven by the amplitude–mass link.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z ** 2 / n; c = (p + z ** 2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return (max(0, c - h), min(1, c + h))
u = use[use.scenario == "C1_null_shuffled"]; c1_rows = []
for (p, c), g in u.groupby(["parameter", "covariate"]):
    k_, n_ = int(g.supported.sum()), len(g); lo, hi = wilson(k_, n_)
    rv, rsd = REAL_SLOPE[(p, c)]; nm, ns = g.slope_median.mean(), g.slope_median.std(ddof=1)
    zc = (rv - nm) / np.sqrt(ns ** 2 + rsd ** 2) if ns > 0 else np.nan
    tcut = float(sst.t.ppf(1 - ALPHA_P / 2, n_ - 1)) if n_ > 1 else np.nan
    tail = (1 + int((np.abs(g.slope_median - nm) >= abs(rv - nm)).sum())) / (n_ + 1)
    a1m, a1s, a1n = A1_NULL.get((p, c), (np.nan, np.nan, 0))
    c1_rows.append({"parameter": p, "covariate": c, "n_mocks": n_, "n_supported": k_, "false_positive_rate": k_ / n_ if n_ else np.nan, "fp_95_low": lo, "fp_95_high": hi,
                    "null_mean": nm, "null_sd": ns, "A1_null_mean_nb06": a1m, "A1_null_sd_nb06": a1s, "A1_n_nb06": a1n,
                    "real_slope": rv, "real_posterior_sd": rsd, "noise_calibrated_z": zc, "t_cutoff": tcut, "normal_cutoff": ZCRIT,
                    "empirical_two_sided_p": tail, "p_floor": 1 / (n_ + 1),
                    "supported_normal_cutoff": bool(np.isfinite(zc) and abs(zc) > ZCRIT), "supported_t_cutoff": bool(np.isfinite(zc) and abs(zc) > tcut)})
c1 = pd.DataFrame(c1_rows)
full = u.groupby("mock_id").filter(lambda d: d.fit_covariate.nunique() == len(FITS_ONE_AT_A_TIME))
per_mock = full.groupby("mock_id").supported.any(); k_, n_ = int(per_mock.sum()), len(per_mock); lo, hi = wilson(k_, n_)
fw_c1 = pd.DataFrame([{"scenario": "C1_null_shuffled", "n_mocks": n_, "n_with_any_supported": k_, "family_wise_rate": k_ / n_ if n_ else np.nan, "fw_95_low": lo, "fw_95_high": hi, "nominal": FAMILY_ALPHA}])
print("Family-wise false-positive rate under the shuffled null:"); display(fw_c1.round(3))
print("Per-slope results under the shuffled null:"); display(c1.round(4))
c1.to_csv(OUT["tables"] / "08_C1_shuffled_null.csv", index=False); fw_c1.to_csv(OUT["tables"] / "08_C1_family_wise.csv", index=False)

## 10. C2 — Recovery Under Correlated Noise

For each injected slope: the recovered median against the injected value, the 95% interval coverage, and two powers:
* **naive power:** the fraction that passes the Notebook 05 rule (|z| > 2.87), as in Notebook 06;
* **calibrated power:** the fraction whose noise-calibrated z against Notebook 06's A1 null, (slope − A1 mean) / √(A1 SD² + posterior SD²), exceeds the Student-t Bonferroni cutoff for the A1 sample size. This is the power of the test actually used in the paper.

Only slopes of the injected covariate are evaluated. With two mocks per cell these are indications, not calibrated detection limits.

In [ ]:
b = use[use.scenario.str.startswith("C2_inject")].copy(); rec = pd.DataFrame()
if len(b):
    b["amplitude"] = b.scenario.str.extract(r"C2_inject_corr_([0-9.]+)x").astype(float)
    b["covered95"] = (b.ci95_low <= b.injected_slope) & (b.injected_slope <= b.ci95_high)
    def cal(r):
        m_, s_, n_ = A1_NULL.get((r.parameter, r.covariate), (np.nan, np.nan, 0))
        zc = (r.slope_median - m_) / np.sqrt(s_ ** 2 + r.slope_sd ** 2) if n_ > 1 else np.nan
        return pd.Series({"z_cal_vs_A1": zc, "t_cutoff_A1": float(sst.t.ppf(1 - ALPHA_P / 2, n_ - 1)) if n_ > 1 else np.nan})
    b = pd.concat([b, b.apply(cal, axis=1)], axis=1)
    b["supported_calibrated"] = b.z_cal_vs_A1.abs() > b.t_cutoff_A1
    b["sign_correct"] = np.sign(b.slope_median) == np.sign(b.injected_slope)
    rec = b.groupby(["covariate", "parameter", "amplitude"]).agg(n=("mock_id", "size"), injected=("injected_slope", "mean"), recovered_mean=("slope_median", "mean"),
                                                                 coverage95=("covered95", "mean"), naive_power=("supported", "mean"),
                                                                 calibrated_power=("supported_calibrated", "mean"), sign_correct=("sign_correct", "mean")).reset_index()
    display(rec.round(4)); rec.to_csv(OUT["tables"] / "08_C2_recovery_correlated.csv", index=False)
    by_param = rec.groupby(["parameter", "amplitude"])[["naive_power", "calibrated_power", "coverage95"]].mean().reset_index()
    print("Power by parameter and amplitude (mean over covariates):"); display(by_param.round(3))
    by_param.to_csv(OUT["tables"] / "08_C2_power_by_parameter.csv", index=False)

## 11. Figures

In [ ]:
def savefig(fig, name):
    p = safe_path(OUT["figures"] / name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.show(); return p
if len(c1):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    lab = [f"{r.parameter} vs {r.covariate}" for r in c1.itertuples()]; y = np.arange(len(c1))
    ax[0].errorbar(c1.null_mean, y - 0.12, xerr=c1.null_sd, fmt="o", label="C1 shuffled null (this run)")
    ax[0].errorbar(c1.A1_null_mean_nb06, y + 0.12, xerr=c1.A1_null_sd_nb06, fmt="s", label="A1 null (Notebook 06)")
    ax[0].scatter(c1.real_slope, y, marker="*", c="r", s=60, zorder=3, label="real slope (Notebook 05)")
    ax[0].axvline(0, c="k", lw=0.6); ax[0].set_yticks(y); ax[0].set_yticklabels(lab, fontsize=8); ax[0].set_xlabel("slope per SD"); ax[0].legend(fontsize=7)
    ax[1].bar(y, c1.false_positive_rate); ax[1].axhline(ALPHA_P, c="k", ls="--", lw=0.8); ax[1].set_xticks(y); ax[1].set_xticklabels(lab, rotation=75, fontsize=7)
    ax[1].set_ylabel("false-positive rate (naive rule)"); fig.suptitle("C1: shuffled-amplitude null"); fig.tight_layout(); savefig(fig, "08_C1_shuffled_null.png")
if len(rec):
    fig, ax = plt.subplots(1, len(POP_PARAMS), figsize=(17, 3.6))
    for a_, p in zip(ax, POP_PARAMS):
        t = b[b.parameter == p]
        for c, col in zip(COVARIATES, ("C0", "C1", "C2")):
            tt = t[t.covariate == c]; a_.errorbar(tt.injected_slope, tt.slope_median, yerr=tt.slope_sd, fmt="o", color=col, label=c, ms=4)
        lim = [min(t.injected_slope.min(), t.slope_median.min()) - 0.05, max(t.injected_slope.max(), t.slope_median.max()) + 0.05]
        a_.plot(lim, lim, "k--", lw=0.8); a_.set_xlabel("injected slope per SD"); a_.set_ylabel("recovered"); a_.set_title(p)
    ax[0].legend(fontsize=7); fig.suptitle("C2: recovery of injected slopes under correlated noise"); fig.tight_layout(); savefig(fig, "08_C2_recovery_correlated.png")

## 12. Exports and Report

In [ ]:
pd.DataFrame(CHECKS).to_csv(OUT["diagnostics"] / "08_integrity_checks.csv", index=False)
config = {"run_id": RUN_ID, "created_utc": RUN_TIMESTAMP, "versions": VERSIONS, "notebook03_run_id": RUN03, "notebook04_D_posterior": str(D_NC),
          "notebook05_slopes": str(runs5[-1]), "notebook06_A1_null": str(runs6[-1]), "truth": TRUTH_CORR,
          "plan": {"N_SHUF": N_SHUF, "C2_AMPLITUDES": C2_AMPLITUDES, "N_C2": N_C2, "mock_seed": MOCK_SEED},
          "mock_fit": MOCK_FIT, "mock_usability": {"rhat_max": MOCK_RHAT_MAX, "ess_min": MOCK_ESS_MIN, "divergences": 0},
          "decision_rule": {"family": N_FAMILY, "per_test_alpha": ALPHA_P, "z_crit_normal": ZCRIT}, "resume_from": RESUME_FROM}
(OUT["config"] / "08_extended_config.json").write_text(json.dumps(config, indent=2, default=str))
pd.DataFrame([{"package": k, "version": str(v)} for k, v in VERSIONS.items()]).to_csv(OUT["config"] / "08_package_versions.csv", index=False)
chk = pd.DataFrame(CHECKS)
c1_txt = "\n".join(f"* {r.parameter} vs {r.covariate}: FP {r.n_supported}/{r.n_mocks}; C1 null {r.null_mean:+.4f} ± {r.null_sd:.4f} (A1 {r.A1_null_mean_nb06:+.4f} ± {r.A1_null_sd_nb06:.4f}); "
                    f"real {r.real_slope:+.4f}; z_cal {r.noise_calibrated_z:+.2f} (normal cut {r.normal_cutoff:.2f}, t cut {r.t_cutoff:.2f}); empirical p {max(r.empirical_two_sided_p, r.p_floor):.3f}"
                    for r in c1.itertuples()) if len(c1) else "* (no usable C1 mocks)"
c2_txt = "\n".join(f"* {r.parameter} vs {r.covariate} at {r.amplitude:.0f}x: injected {r.injected:+.4f}, recovered {r.recovered_mean:+.4f}, coverage {r.coverage95:.2f}, "
                    f"naive power {r.naive_power:.2f}, calibrated power {r.calibrated_power:.2f} (n={r.n})" for r in rec.itertuples()) if len(rec) else "* (no usable C2 mocks)"
report = f"""# Notebook 08 — Shuffled-Amplitude Null and Recovery Under Correlated Noise — Report

*Run {RUN_ID} ({RUN_TIMESTAMP} UTC). Notebook 04 Model D posterior {D_NC.parents[1].name}; Notebook 05 run {runs5[-1].parents[1].name}; Notebook 06 run {runs6[-1].parents[1].name}. This file exists only if the notebook ran.*

## Setup
* Truth: universal RAR at Model D medians {json.dumps({k: round(v, 4) for k, v in TRUTH_CORR.items()})}; per-galaxy correlated noise with Model D A_i and l_i.
* C1: A_i permuted across galaxies in every mock (corr(ln A, logMbar) {corrA['logMbar']:+.2f} in Model D; after shuffling {np.mean(shuf_corr):+.3f} ± {np.std(shuf_corr):.3f}).
* C2: Notebook 05 slopes injected at {C2_AMPLITUDES} x on unshuffled correlated noise; {N_C2} mocks per covariate and amplitude.
* Mock fits: {MOCK_FIT['chains']} chains x ({MOCK_FIT['warmup']} + {MOCK_FIT['samples']}); usable if R-hat < {MOCK_RHAT_MAX}, ESS >= {MOCK_ESS_MIN}, 0 divergences. Usable fractions: {summary_use.round(3).to_dict("records")}

## C1 — shuffled-amplitude null
* Family-wise: {fw_c1.n_with_any_supported.iloc[0] if len(fw_c1) else 'n/a'}/{fw_c1.n_mocks.iloc[0] if len(fw_c1) else 'n/a'} mocks with >= 1 naive 'supported' slope (nominal {FAMILY_ALPHA:.0%}).
{c1_txt}

## C2 — recovery under correlated noise
{c2_txt}

## Interpretation rules (declared before running)
* If the C1 null reproduces the A1 null (similar means, SDs and false-positive rates), the Notebook 06 calibration was **not** driven by the amplitude–mass link, and the circularity concern is answered.
* If the C1 null has much smaller spurious slopes than A1, then part of A1's spurious signal came from the mass-dependent amplitude; real slopes must then be compared with C1, and any slope beyond the C1 t-cutoff is reported as a candidate population dependence (or mass-dependent noise; the two cannot be separated without GP fits).
* Calibrated power in C2 is the probability that the paper's test detects a dependence as large as the measured one (1x) or twice as large (2x) in the presence of correlated noise. Low calibrated power means the paper's null results are uninformative for that parameter and must be stated as such.
* With N mocks the empirical p cannot go below 1/(N+1); the t-cutoff accounts for estimating the null SD from N mocks.

## Checks
{len(chk)} checks: {int((chk.status == "FAIL").sum())} FAIL, {int((chk.status == "WARN").sum())} WARN.
"""
rep_p = OUT["tables"] / "08_Extended_Injection_Report.md"; rep_p.write_text(report, encoding="utf-8")
manifest = pd.DataFrame([{"file": str(p.relative_to(OUT_ROOT)), "bytes": p.stat().st_size} for p in sorted(OUT_ROOT.rglob("*")) if p.is_file()])
manifest.to_csv(OUT["config"] / "08_output_manifest.csv", index=False)
print(report); print("Outputs:", OUT_ROOT)